# ReelRun

### Model loading-- this will take 10-15 mins(only once) dont run this cell multiple times

In [ ]:
# Install dependencies
!pip install -q diffusers transformers accelerate safetensors

import torch
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

# Load pipeline
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
).to("cuda")

# Disable safety checker (saves memory + speed)
pipe.safety_checker = None

# Enable memory optimizations
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()

# Faster scheduler
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config
)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2263: FutureWarning: `enable_vae_slicing` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_slicing()` on a `StableDiffusionXLPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_slicing()`.
  deprecate(


### Use narration for audio, rest are for the image generator

In [ ]:
import json

keyword = "Magnetic Hill Ladakh"

def generate_content(keyword):

    # In real system this comes from your LLM
    response = {
        "topic": "The Whispering Forest of Meghalaya",

        "narration": """
Deep in the misty hills of Meghalaya lies a forest where locals say the trees whisper at night.
Travelers walking through the dense jungle often hear soft murmurs carried by the wind.
Some believe it is the forest spirits speaking.
But scientists discovered something fascinating.
When strong winds pass through the hollow bamboo and thick tree roots, they create natural resonating sounds that resemble whispers.
What sounds like a supernatural mystery is actually nature playing music through the forest.
""",

        "image_prompts": [
            "dense misty forest in Meghalaya India, tall ancient trees covered with moss, thick fog drifting between trunks, cinematic lighting, ultra realistic, 8k nature photography, mysterious atmosphere",

            "narrow jungle trail through a foggy Meghalaya forest, traveler with backpack walking alone, soft morning mist, beams of sunlight piercing through leaves, cinematic composition, National Geographic style",

            "close view of hollow bamboo and tree roots in a rainforest producing wind sounds, lush green textures, droplets of water on leaves, macro cinematic nature shot",

            "wide aerial drone shot of Meghalaya rainforest hills covered in dense fog and clouds, dramatic landscape, deep green forest stretching to horizon, cinematic documentary style",

            "mysterious night scene inside a misty forest, moonlight shining through tall trees, fog glowing softly, magical and eerie atmosphere, ultra detailed cinematic lighting"
        ]
    }

    return response


result = generate_content(keyword)

print(json.dumps(result, indent=2))

{
  "topic": "The Whispering Forest of Meghalaya",
  "narration": "\nDeep in the misty hills of Meghalaya lies a forest where locals say the trees whisper at night.\nTravelers walking through the dense jungle often hear soft murmurs carried by the wind.\nSome believe it is the forest spirits speaking.\nBut scientists discovered something fascinating.\nWhen strong winds pass through the hollow bamboo and thick tree roots, they create natural resonating sounds that resemble whispers.\nWhat sounds like a supernatural mystery is actually nature playing music through the forest.\n",
  "image_prompts": [
    "dense misty forest in Meghalaya India, tall ancient trees covered with moss, thick fog drifting between trunks, cinematic lighting, ultra realistic, 8k nature photography, mysterious atmosphere",
    "narrow jungle trail through a foggy Meghalaya forest, traveler with backpack walking alone, soft morning mist, beams of sunlight piercing through leaves, cinematic composition, National Ge

In [ ]:
anchor = """
ancient moss covered trees, hollow bamboo stems, thick rainforest vegetation,
wet leaves and twisted roots, mist flowing through the forest
"""

environment = """
Whispering Forest Meghalaya India, dense subtropical rainforest,
rolling fog between tall trees, mysterious natural atmosphere
"""

style = [
    "8k",
    "cinematic lighting",
    "35mm lens",
    "ultra realistic",
    "National Geographic documentary photography",
    "volumetric light",
    "dramatic atmosphere"
]

style_block = ", ".join(style)

prompts = [

    f"wide establishing shot of a misty rainforest valley, {anchor}, {environment}, {style_block}",

    f"medium shot of a narrow jungle trail with a lone traveler walking through foggy forest, {anchor}, {environment}, {style_block}",

    f"close cinematic shot of hollow bamboo and tree roots as wind passes through them creating whispering sounds, {anchor}, {environment}, {style_block}",

    f"aerial drone shot above the dense Meghalaya rainforest covered in drifting mist and clouds, {anchor}, {environment}, {style_block}",

    f"mysterious night scene inside the forest illuminated by moonlight and glowing fog, {anchor}, {environment}, {style_block}"
]

In [ ]:
import torch
images = []
seed = 42
for prompt in result["image_prompts"]:
    generator = torch.Generator(device="cuda").manual_seed(seed)
    img = pipe(
        prompt,
        num_inference_steps=25,
        guidance_scale=6.5,
        width=1024,
        height=1024,
        generator=generator
    ).images[0]
    images.append(img)

    seed += 7

  0%|          | 0/25 [00:00<?, ?it/s]

  deprecate(



  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
import os

os.makedirs("frames", exist_ok=True)
for i, img in enumerate(images):
    img.save(f"frames/frame_{i}.png")

In [ ]:
import cv2

def create_pan_clip(image_path, output_path, duration=6, fps=30):

    img = cv2.imread(image_path)
    h, w, _ = img.shape

    window_w = int(w * 0.7)
    total_frames = duration * fps

    video = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (1080,1920)
    )

    for i in range(total_frames):

        x = int((w - window_w) * (i / total_frames))

        crop = img[:, x:x+window_w]
        frame = cv2.resize(crop, (1080,1920))

        video.write(frame)

    video.release()

In [ ]:
clips = []

for i in range(len(images)):

    input_img = f"frames/frame_{i}.png"
    output_vid = f"clip_{i}.mp4"

    create_pan_clip(input_img, output_vid)

    clips.append(output_vid)

In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips

video_clips = [VideoFileClip(c) for c in clips]

final_video = concatenate_videoclips(video_clips)

final_video.write_videofile("scenes.mp4", fps=30)

t:  82%|████████▏ | 741/900 [02:02<00:26,  6.08it/s, now=None]

Moviepy - Building video scenes.mp4.
Moviepy - Writing video scenes.mp4




t:  82%|████████▏ | 741/900 [03:58<00:26,  6.08it/s, now=None]

Moviepy - Done !
Moviepy - video ready scenes.mp4


### Video generated as scenes.mp4

## Now we generate audio + subtitles

In [ ]:
import edge_tts
from moviepy.editor import AudioFileClip

# narration = response["narration"]
narration = """
Deep in the misty hills of Meghalaya lies a forest where locals say the trees whisper at night.
Travelers walking through the dense jungle often hear soft murmurs carried by the wind.
Some believe it is the forest spirits speaking.
But scientists discovered something fascinating.
When strong winds pass through the hollow bamboo and thick tree roots, they create natural resonating sounds that resemble whispers.
What sounds like a supernatural mystery is actually nature playing music through the forest.
"""

async def generate_audio(text):
    communicate = edge_tts.Communicate(text, "en-US-ChristopherNeural")
    await communicate.save("voice.mp3")

await generate_audio(narration)

print("Audio generated: voice.mp3")

Audio generated: voice.mp3


In [ ]:
from faster_whisper import WhisperModel

audio_path = "voice.mp3"

# tiny = fast + light
model = WhisperModel("tiny", device="cpu", compute_type="int8")

segments, _ = model.transcribe(
    audio_path,
    word_timestamps=True
)

words_data = []

for segment in segments:
    for w in segment.words:
        words_data.append((w.start, w.end, w.word.strip()))

# save timestamps
with open("timestamps.txt", "w", encoding="utf-8") as f:
    for s,e,w in words_data:
        f.write(f"{s:.2f}|{e:.2f}|{w}\n")

print("✅ Timestamps saved to timestamps.txt")


✅ Timestamps saved to timestamps.txt


In [ ]:
video = VideoFileClip("scenes.mp4")
audio = AudioFileClip("voice.mp3")

video = video.set_audio(audio)

In [ ]:
from moviepy.editor import VideoFileClip, AudioFileClip, ImageClip, CompositeVideoClip
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import re

video = VideoFileClip("scenes.mp4")
audio = AudioFileClip("voice.mp3")
video = video.set_audio(audio)

danger_words = {
    "danger", "death", "kill", "warning",
    "risk", "scary", "fear", "dead"
}

def get_color(word):
    if re.search(r"\d", word):
        return "#4CFF00"
    if word.lower() in danger_words:
        return "#FF3B3B"
    return "#FFD93D"

def text_img(text):
    W, H = 520, 150
    img = Image.new("RGBA", (W, H), (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)

    # Colab-safe font
    font = ImageFont.truetype(
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    80
)

    color = get_color(text)

    draw.rounded_rectangle(
        [(0, 0), (W, H)],
        radius=40,
        fill=(0, 0, 0, 180)
    )

    bbox = draw.textbbox((0, 0), text, font=font)
    w = bbox[2] - bbox[0]
    h = bbox[3] - bbox[1]

    x = (W - w) // 2
    y = (H - h) // 2

    for dx in range(-3, 4):
        for dy in range(-3, 4):
            draw.text((x + dx, y + dy), text, font=font, fill="black")

    draw.text((x, y), text, font=font, fill=color)

    return np.array(img)

def word_clip(word, start, end):
    img = text_img(word.upper())
    dur = end - start

    return (
        ImageClip(img)
        .set_start(start)
        .set_duration(dur)
        .set_position(("center", 0.5), relative=True)
        .resize(lambda t: 1 + 0.35 * np.exp(-6 * t))
    )

subs = []

with open("timestamps.txt", "r", encoding="utf-8") as f:
    for line in f:
        s, e, w = line.strip().split("|")
        s, e = float(s), float(e)
        subs.append(word_clip(w, s, e))

final = CompositeVideoClip([video] + subs)

final.write_videofile(
    "final_reel.mp4",
    fps=24,
    codec="libx264",
    audio_codec="aac"
)

Moviepy - Building video final_reel.mp4.
MoviePy - Writing audio in final_reelTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video final_reel.mp4



Moviepy - Done !
Moviepy - video ready final_reel.mp4


In [ ]:
from google.colab import files
files.download("final_reel.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>